# TMS AI - Data Exploration and Initial Analysis

**Embedded AI for Offline Delay Prediction in Transportation Management Systems**

This notebook explores the taxi trajectory dataset and prepares it for delay prediction modeling.

---

## Objectives
1. Download and load GPS trajectory data from Kaggle
2. Explore data structure and quality
3. Visualize trajectories and patterns
4. Extract features for delay prediction
5. Prepare data for baseline and deep learning models

In [ ]:
# TMS AI - Data Exploration and Initial Analysis

This notebook explores the taxi trajectory dataset and prepares it for delay prediction modeling.

## Setup and Imports

## 1. Environment Setup and Imports

In [ ]:
# Core libraries
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("✅ Environment setup complete")
print(f"📂 Project root: {PROJECT_ROOT}")
print(f"📂 Raw data: {DATA_RAW}")
print(f"📂 Processed data: {DATA_PROCESSED}")

## 2. Kaggle API Configuration and Dataset Download

First, verify that Kaggle API is properly configured and download the dataset.

In [ ]:
# Import Kaggle API
from kaggle.api.kaggle_api_extended import KaggleApi

# Authenticate
api = KaggleApi()
api.authenticate()

print("✅ Kaggle API authenticated successfully")

In [ ]:
# List available taxi/trajectory datasets
# Common options: crailtap/taxi-trajectory, jsyousef/data-of-10000-taxi-trips-in-portugal

DATASET = "crailtap/taxi-trajectory"  # Porto Taxi Trajectory Dataset

# List files in dataset
print(f"📂 Files in dataset '{DATASET}':\n")
files = api.dataset_list_files(DATASET).files
for i, f in enumerate(files, 1):
    size_mb = f.size / (1024 * 1024)
    print(f"{i}. {f.name:<40} ({size_mb:.2f} MB)")

In [ ]:
# Download dataset (if not already downloaded)
if not any(DATA_RAW.glob("*.csv")):
    print("⬇️  Downloading dataset...")
    api.dataset_download_files(DATASET, path=str(DATA_RAW), unzip=True)
    print("✅ Dataset downloaded")
else:
    print("✅ Dataset already exists locally")

# List downloaded files
print("\n📋 Downloaded files:")
for f in DATA_RAW.iterdir():
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  - {f.name} ({size_mb:.2f} MB)")

## 3. Load and Explore Data

Load the dataset and perform initial exploration.

In [ ]:
# Find CSV files
csv_files = list(DATA_RAW.glob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s)")

# Load the first CSV
if csv_files:
    df = pd.read_csv(csv_files[0])
    print(f"\n✅ Loaded: {csv_files[0].name}")
    print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
else:
    print("⚠️  No CSV files found. Please run the download cell above.")

In [ ]:
# Display first few rows
df.head(10)

In [ ]:
# Data info
print("📋 Dataset Info:")
print("=" * 60)
df.info()
print("\n" + "=" * 60)

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

print("🔍 Missing Values:")
print("=" * 60)
for col, count in missing[missing > 0].items():
    print(f"{col:<30} {count:>8,} ({missing_pct[col]:>6.2f}%)")
    
if missing.sum() == 0:
    print("✅ No missing values found")

## 4. Parse GPS Trajectories

The Porto Taxi dataset stores GPS coordinates as JSON-encoded polylines. We'll parse these and extract features.

In [ ]:
def parse_polyline(polyline_str):
    """Parse GPS polyline from JSON string"""
    try:
        coords = json.loads(polyline_str)
        return [(lat, lon) for lon, lat in coords]  # Note: dataset uses [lon, lat]
    except:
        return []

# Parse a sample trajectory
if 'POLYLINE' in df.columns:
    sample_polyline = df.iloc[0]['POLYLINE']
    sample_trajectory = parse_polyline(sample_polyline)
    
    print(f"📍 Sample trajectory:")
    print(f"   Number of GPS points: {len(sample_trajectory)}")
    if sample_trajectory:
        print(f"   First point: {sample_trajectory[0]}")
        print(f"   Last point: {sample_trajectory[-1]}")
else:
    print("⚠️  No POLYLINE column found")

In [ ]:
# Calculate trajectory lengths
if 'POLYLINE' in df.columns:
    df['trajectory_length'] = df['POLYLINE'].apply(lambda x: len(parse_polyline(x)))
    
    print("📊 Trajectory Length Statistics:")
    print(df['trajectory_length'].describe())
    
    # Visualize distribution
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    df['trajectory_length'].hist(bins=50, edgecolor='black')
    plt.xlabel('Number of GPS Points')
    plt.ylabel('Frequency')
    plt.title('Distribution of Trajectory Lengths')
    plt.grid(alpha=0.3)
    
    plt.subplot(1, 2, 2)
    df['trajectory_length'].plot(kind='box', vert=False)
    plt.xlabel('Number of GPS Points')
    plt.title('Trajectory Length Box Plot')
    plt.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Analyze Trip Duration and Temporal Patterns

In [ ]:
# Convert timestamp to datetime
if 'TIMESTAMP' in df.columns:
    df['datetime'] = pd.to_datetime(df['TIMESTAMP'], unit='s')
    df['hour'] = df['datetime'].dt.hour
    df['day_of_week'] = df['datetime'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    print("✅ Temporal features extracted")
    print(f"   Date range: {df['datetime'].min()} to {df['datetime'].max()}")

In [ ]:
# Visualize temporal patterns
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Trips by hour
if 'hour' in df.columns:
    df['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0, 0], color='steelblue')
    axes[0, 0].set_xlabel('Hour of Day')
    axes[0, 0].set_ylabel('Number of Trips')
    axes[0, 0].set_title('Trips by Hour of Day')
    axes[0, 0].grid(alpha=0.3)

# Trips by day of week
if 'day_of_week' in df.columns:
    day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    df['day_of_week'].value_counts().sort_index().plot(kind='bar', ax=axes[0, 1], color='coral')
    axes[0, 1].set_xticklabels(day_names, rotation=45)
    axes[0, 1].set_xlabel('Day of Week')
    axes[0, 1].set_ylabel('Number of Trips')
    axes[0, 1].set_title('Trips by Day of Week')
    axes[0, 1].grid(alpha=0.3)

# Travel time distribution
if 'TRAVEL_TIME' in df.columns:
    # Filter outliers for better visualization
    travel_time_filtered = df[df['TRAVEL_TIME'] < df['TRAVEL_TIME'].quantile(0.95)]
    travel_time_filtered['TRAVEL_TIME'].hist(bins=50, ax=axes[1, 0], edgecolor='black', color='green', alpha=0.7)
    axes[1, 0].set_xlabel('Travel Time (seconds)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Travel Time Distribution (95th percentile)')
    axes[1, 0].grid(alpha=0.3)

# Weekend vs Weekday comparison
if 'is_weekend' in df.columns and 'TRAVEL_TIME' in df.columns:
    weekend_data = df[df['is_weekend'] == 1]['TRAVEL_TIME']
    weekday_data = df[df['is_weekend'] == 0]['TRAVEL_TIME']
    
    axes[1, 1].boxplot([weekday_data, weekend_data], labels=['Weekday', 'Weekend'])
    axes[1, 1].set_ylabel('Travel Time (seconds)')
    axes[1, 1].set_title('Travel Time: Weekday vs Weekend')
    axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Feature Engineering and Delay Labeling

Calculate speed, distance, and label trips as delayed based on ETA deviation.

In [ ]:
def calculate_haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance between two GPS points in kilometers"""
    R = 6371  # Earth radius in km
    
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

def calculate_total_distance(polyline_str):
    """Calculate total trip distance from polyline"""
    trajectory = parse_polyline(polyline_str)
    if len(trajectory) < 2:
        return 0
    
    total_distance = 0
    for i in range(1, len(trajectory)):
        lat1, lon1 = trajectory[i-1]
        lat2, lon2 = trajectory[i]
        total_distance += calculate_haversine_distance(lat1, lon1, lat2, lon2)
    
    return total_distance

# Calculate distances
if 'POLYLINE' in df.columns:
    print("🔧 Calculating distances...")
    # Use sample for faster processing (remove .head(1000) for full dataset)
    sample_df = df.head(1000).copy()
    sample_df['distance_km'] = sample_df['POLYLINE'].apply(calculate_total_distance)
    print("✅ Distances calculated")

In [ ]:
# Calculate average speed
if 'TRAVEL_TIME' in sample_df.columns and 'distance_km' in sample_df.columns:
    sample_df['avg_speed_kmh'] = (sample_df['distance_km'] / sample_df['TRAVEL_TIME']) * 3600
    
    # Filter invalid speeds
    sample_df = sample_df[sample_df['avg_speed_kmh'] < 150]  # Remove unrealistic speeds
    
    print("📊 Speed Statistics:")
    print(sample_df['avg_speed_kmh'].describe())
    
    # Visualize speed distribution
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    sample_df['avg_speed_kmh'].hist(bins=50, edgecolor='black', color='purple', alpha=0.7)
    plt.xlabel('Average Speed (km/h)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Average Speeds')
    plt.grid(alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.scatter(sample_df['distance_km'], sample_df['TRAVEL_TIME'], alpha=0.3, s=10)
    plt.xlabel('Distance (km)')
    plt.ylabel('Travel Time (seconds)')
    plt.title('Distance vs Travel Time')
    plt.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Label delays based on ETA deviation
EXPECTED_AVG_SPEED = 30  # km/h (typical city driving)
DELAY_THRESHOLD_MINUTES = 15  # minutes

if 'distance_km' in sample_df.columns and 'TRAVEL_TIME' in sample_df.columns:
    # Calculate expected travel time
    sample_df['expected_time_sec'] = (sample_df['distance_km'] / EXPECTED_AVG_SPEED) * 3600
    
    # Calculate delay
    sample_df['delay_sec'] = sample_df['TRAVEL_TIME'] - sample_df['expected_time_sec']
    sample_df['delay_min'] = sample_df['delay_sec'] / 60
    
    # Binary delay label
    sample_df['is_delayed'] = (sample_df['delay_min'] > DELAY_THRESHOLD_MINUTES).astype(int)
    
    print(f"📊 Delay Statistics:")
    print(f"   Total trips: {len(sample_df):,}")
    print(f"   Delayed trips: {sample_df['is_delayed'].sum():,} ({sample_df['is_delayed'].mean():.1%})")
    print(f"   On-time trips: {(1 - sample_df['is_delayed']).sum():,} ({(1 - sample_df['is_delayed'].mean()):.1%})")
    
    # Visualize delay distribution
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    sample_df['delay_min'].hist(bins=50, edgecolor='black', color='orange', alpha=0.7)
    plt.axvline(DELAY_THRESHOLD_MINUTES, color='red', linestyle='--', label=f'Threshold ({DELAY_THRESHOLD_MINUTES} min)')
    plt.xlabel('Delay (minutes)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Delays')
    plt.legend()
    plt.grid(alpha=0.3)
    
    plt.subplot(1, 2, 2)
    sample_df['is_delayed'].value_counts().plot(kind='bar', color=['green', 'red'])
    plt.xticks([0, 1], ['On-Time', 'Delayed'], rotation=0)
    plt.ylabel('Count')
    plt.title('Delay vs On-Time Trips')
    plt.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. Summary and Next Steps

### Key Findings
- Dataset contains GPS trajectory data with timestamps and travel times
- Temporal patterns show peak hours and weekday/weekend differences
- Speed and distance features can be extracted from GPS polylines
- Delay labeling based on ETA deviation identifies delayed vs on-time trips

### Next Steps
1. ✅ Run `scripts/download_kaggle_dataset.py` to download full dataset
2. ✅ Run `scripts/preprocess_data.py` to create time-series sequences
3. ⏭️ Run `scripts/train_baseline.py` to train Logistic Regression and Random Forest models
4. ⏭️ Build LSTM/GRU model for deep learning approach
5. ⏭️ Convert model to ONNX/TFLite for edge deployment
6. ⏭️ Implement OSMnx routing and visualization
7. ⏭️ Build Flask/FastAPI backend
8. ⏭️ Create mobile app UI

In [ ]:
# Save sample processed data
if 'sample_df' in locals():
    sample_output_path = DATA_PROCESSED / "sample_features.csv"
    sample_df.to_csv(sample_output_path, index=False)
    print(f"✅ Sample data saved to: {sample_output_path}")